## 导入

本 notebook 是 **Week 3** 作业：用 LLM 当「数据集规划师」，先产出 JSON 规格（spec），再用 NumPy/Pandas 按规格合成数据；接着把流水线注册成 **Function Calling / Tool**，用 Gradio 聊天驱动；最后对比多个 Hugging Face 开源模型直接吐 JSON 的能力。

**怎么跑（前半：OpenAI + Gradio）**

1. 在 Google Colab Secrets 配置 `OPENAI_API_KEY`（后半还需要 `HF_TOKEN`）。
2. 从上到下运行：定义 system schema → 实现生成列/任务处理 → `dataset_pipeline` → 注册工具 → 启动聊天界面。


In [ ]:
# ========== 导入：Colab 密钥、OpenAI、Gradio ==========

# os：通用系统接口（本格主要占位，后续可能用到）
import os
# json：解析 LLM 返回的 JSON、序列化 tool 结果
import json
# Colab Secrets：安全读取 OPENAI_API_KEY / HF_TOKEN
from google.colab import userdata
# OpenAI 客户端：Chat Completions + tools
from openai import OpenAI
# Gradio：聊天 + 预览表 + 下载
import gradio as gr


## 定义工具

下面的 `system_message` 是发给「规划师」模型的 **JSON Schema 约定**：只许返回合法 JSON，并按回归 / 分类 / 聚类任务填不同字段。  
**注意：整段英文 prompt 是可运行指令，不要翻译。**


In [ ]:
# ========== 规划师 system prompt：强制只返回符合 Schema 的 JSON ==========
# 下面三引号里的英文是发给模型的指令与 Schema，改译会破坏工具行为，必须原样保留

system_message = """
You are a dataset generation planner.

Return ONLY valid JSON.

Schema:

{
  "task": "regression | classification | clustering",
  "num_rows": int,
  "columns": [
    {
      "name": string,
      "type": "int" | "float" | "categorical" | "text" | "boolean",
      "params": {
        // int/float → {"min": number, "max": number}
        // categorical → {"values": [list of strings]}
        // text → {"pattern": string}
        // boolean → {}
      }
    }
  ],
  "target": string or null,
  "target_type": "float" | "categorical" | null,
  "relationships": [
    "expressions like: price = year*300 - mileage*200 + noise(0,1000)"
  ],
  "class_distribution": {
    "class_name": probability
  },
  "cluster_config": {
    "num_clusters": int,
    "method": "gaussian" | "separable"
  }
}

Rules:

- Always include num_rows
- Always include at least 2–5 columns

TASK RULES:

1. Regression:
   - target MUST be present
   - target_type MUST be "float"
   - MUST include at least one relationship

2. Classification:
   - target MUST be present
   - target_type MUST be "categorical"
   - MUST include class_distribution
   - class_distribution values should sum to ~1

3. Clustering:
   - target MUST be null
   - target_type MUST be null
   - MUST include cluster_config
   - cluster_config.num_clusters should be 2–6

GENERAL RULES:

- Infer realistic feature names and ranges
- Ensure relationships use only defined columns
- Do NOT leave fields empty
- Use reasonable defaults if user input is incomplete

ONLY return valid JSON. NO explanations.
"""


In [ ]:
# ========== 调用 GPT：把自然语言需求变成 JSON spec ==========

# 再次导入（本格可独立重跑）
import json
from openai import OpenAI
# 从 Colab Secrets 取 OpenAI 密钥
openai_api_key = userdata.get('OPENAI_API_KEY')
# 创建客户端
client = OpenAI(api_key = openai_api_key)

def llm_generate_spec(user_input):
    # Chat Completions：system 定 Schema，user 放自然语言需求
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_input}
        ]
    )
    # 先打印原始文本，方便调试模型是否夹带了非 JSON
    print(response.choices[0].message.content)
    # json.loads：字符串 → Python dict（失败会抛异常）
    return json.loads(response.choices[0].message.content)


In [ ]:
# ========== 按列类型用 NumPy 生成一列随机数据 ==========

def generate_column(col, n):
    # 函数内导入 numpy：生成随机数
    import numpy as np

    # 列类型：int / float / categorical / boolean / text
    t = col["type"]
    # params：范围、类别列表等；缺省用空 dict
    p = col.get("params", {})

    if t == "int":
        # 整数均匀采样 [min, max)
        return np.random.randint(p.get("min", 0), p.get("max", 100), n)

    elif t == "float":
        # 浮点均匀采样
        return np.random.uniform(p.get("min", 0), p.get("max", 100), n)

    elif t == "categorical":
        # 从给定取值列表中有放回抽样
        return np.random.choice(p.get("values", ["A", "B"]), n)

    elif t == "boolean":
        # 真假各半（默认等概率）
        return np.random.choice([True, False], n)

    elif t == "text":
        # 简单占位文本：列名_下标
        return [f"{col['name']}_{i}" for i in range(n)]


In [ ]:
# ========== 安全求值：在受限命名空间里计算关系表达式 ==========

def safe_eval(expr, df):
    import numpy as np

    # 允许的「全局」名字：noise(...) 与 np，避免开放 __builtins__
    allowed = {
        "noise": lambda mean, std: np.random.normal(mean, std, len(df)),
        "np": np
    }

    # eval：表达式右侧；局部变量 = 各列 Series + allowed
    # {"__builtins__": {}} 关掉内建，降低任意代码执行风险
    return eval(expr, {"__builtins__": {}}, {**df.to_dict("series"), **allowed})


In [ ]:
# ========== 回归任务：按 relationships 写回目标/派生列 ==========

def handle_regression(df, spec):
    # 逐条 "left = right" 关系式
    for rel in spec.get("relationships", []):
        # 按第一个 = 拆成左右（左侧列名，右侧表达式）
        left, right = rel.split("=")
        # 把求值结果写进 DataFrame 新列/覆盖列
        df[left.strip()] = safe_eval(right.strip(), df)

    return df


In [ ]:
# ========== 分类任务：按 class_distribution 抽样标签 ==========

def handle_classification(df, spec):
    import numpy as np

    # 目标列名
    target = spec["target"]
    # 类别 → 概率；缺省两类各 0.5
    dist = spec.get("class_distribution", {"A": 0.5, "B": 0.5})

    classes = list(dist.keys())
    probs = list(dist.values())

    # 归一化，防止概率和不为 1
    probs = np.array(probs) / np.sum(probs)

    # 按概率生成整列标签
    df[target] = np.random.choice(classes, size=len(df), p=probs)

    return df


In [ ]:
# ========== 聚类/无监督：给数值列加「簇偏移」制造可分结构 ==========

def handle_unsupervised(df, spec):
    import numpy as np

    # 读取聚类配置；缺省空 dict
    cluster_cfg = spec.get("cluster_config", {})
    # 簇数；缺省 3
    num_clusters = cluster_cfg.get("num_clusters", 3)  # fallback

    # 每行随机分配一个簇 id
    cluster_ids = np.random.randint(0, num_clusters, len(df))

    # 只对非 object（非字符串）列加偏移：簇 id * 随机幅度
    for col in df.columns:
        if df[col].dtype != "object":
            df[col] += cluster_ids * np.random.uniform(5, 20)

    return df


In [ ]:
# ========== 执行 spec：先造特征列，再按 task 分支后处理 ==========

def run_spec(spec):
    import pandas as pd

    # 行数
    n = spec["num_rows"]

    data = {}

    # 生成基础特征列（跳过 target，目标列由后续 handler 写入）
    for col in spec["columns"]:
        if col["name"] != spec.get("target"):
            data[col["name"]] = generate_column(col, n)

    # 字典 → DataFrame
    df = pd.DataFrame(data)

    # 任务类型；未知时走无监督分支
    task = spec.get("task", "unsupervised")

    if task == "regression":
        df = handle_regression(df, spec)

    elif task == "classification":
        df = handle_classification(df, spec)

    else:
        df = handle_unsupervised(df, spec)

    return df


In [ ]:
# ========== 端到端流水线：LLM 出 spec → 造表 → 存 CSV ==========

def dataset_pipeline(modified_request: str):
    # 1) 自然语言 → JSON spec
    spec = llm_generate_spec(modified_request)
    # 2) spec → DataFrame
    df = run_spec(spec)

    # 3) 落盘，供下载与后续 pd.read_csv
    file_path = "dataset.csv"
    df.to_csv(file_path, index=False)

    # 返回预览（前几行 dict）与文件路径
    return {
        "preview": df.head().to_dict(),
        "file_path": file_path
    }


## 测试工具

下面用一条「客户分群 / clustering」自然语言请求跑通 `dataset_pipeline`，检查能否得到 CSV 与预览。


In [ ]:
# ========== 冒烟测试：聚类数据集请求（英文用户输入保持原样） ==========

test_run = dataset_pipeline("""Create a clustering dataset with 800 rows for customer segmentation.
Features include income, spending score, gender,City, age.
Generate 4 distinct clusters.""")


In [ ]:
# 查看流水线返回值：应含 preview 与 file_path
test_run


In [ ]:
# ========== 从落盘 CSV 读回，确认表格可打开 ==========

import pandas as pd
# 用上一步返回的路径读 CSV
df = pd.read_csv(test_run['file_path'])
# 看前几行
df.head()


In [ ]:
# 查看形状：(行数, 列数)，应接近请求里的 800 行
df.shape


## 添加工具功能（Function Calling）

把 `dataset_pipeline` 描述成 OpenAI **tools** JSON Schema，之后聊天模型可以用 `tool_calls` 触发真正的数据生成，而不是自己空想一张表。


In [ ]:
# ========== 工具 schema：告诉模型有哪些参数、何时该调用 ==========
# name/description/parameters 里的英文是 API 契约，保持原样

pipeline_function = {
    "name": "dataset_pipeline",
    "description": "Generate a synthetic dataset based on a structured natural language request. The tool creates the dataset, returns a preview of the data, and provides a file path to download the CSV.",
    "parameters": {
        "type": "object",
        "properties": {
            "modified_request": {
                "type": "string",
                "description": "A clear and complete natural language instruction describing the dataset. It should include task type (regression, classification, or clustering), number of rows, feature names, and target variable if applicable."
            }
        },
        "required": ["modified_request"]
    }
}


In [ ]:
# 包装成 Chat Completions 的 tools 列表：type=function + function=上面的 schema
tools = [{"type": "function", "function": pipeline_function}]


In [ ]:
# ========== 执行 tool_calls：解析参数 → 调流水线 → 拼 tool 消息 ==========

def handle_tool_calls(message):
    responses = []
    # 一条 assistant 消息可能带多个 tool_call
    for tool_call in message.tool_calls:
        # 只处理我们注册的 dataset_pipeline
        if tool_call.function.name == "dataset_pipeline":
            # arguments 是 JSON 字符串
            arguments = json.loads(tool_call.function.arguments)
            request = arguments.get('modified_request')
            # 真正生成数据
            data_dict = dataset_pipeline(request)
            # 按 OpenAI 协议回一条 role=tool 的消息
            responses.append({
            "role": "tool",
            "content": json.dumps(data_dict),
            "tool_call_id": tool_call.id
        })
    return responses


## Gradio 界面

「大脑」模型（`BRAIN_SYSTEM_PROMPT`）负责澄清需求并在信息足够时 **调用 tool**；界面展示对话、数据预览与 CSV 下载。


In [ ]:
# ========== 聊天「大脑」system prompt：澄清需求 + 在合适时机调用 tool ==========
# 整段英文是行为规范与正反例，改译会改变何时 tool_call，必须原样保留

BRAIN_SYSTEM_PROMPT = """
You are an assistant that helps users design synthetic datasets.

Your job:
1. Understand the dataset requirements
2. Ask clarifying questions if needed
3. Once sufficient details are available, call the dataset_pipeline tool

-----------------------
REQUIRED INFORMATION
-----------------------

Before calling the tool, ensure the following are known or reasonably inferred:

- Dataset domain (cars, healthcare, finance, etc.)
- Task type:
  - regression (numeric target)
  - classification (categorical target)
  - clustering / unsupervised (no target)
- Number of rows
- Feature names (at least 2–5 relevant features)
- Target variable (if supervised)

Optional but helpful:
- Relationships (e.g., “price depends on mileage and year”)
- Feature types (numeric/categorical)
- Class distribution (for classification)

If required information is missing, ask a clear follow-up question.

-----------------------
TOOL CALL INSTRUCTIONS
-----------------------

When ready, call the dataset_pipeline tool with a SINGLE, well-structured natural language instruction that includes all relevant details.

DO NOT:
- Generate datasets yourself
- Output JSON
- Call the tool with incomplete or vague input

-----------------------
EXAMPLES (VERY IMPORTANT)
-----------------------

BAD tool input (too vague):
"cars dataset"

BAD tool input (missing task and target):
"create dataset with mileage and year"

GOOD tool input (regression):
"Create a regression dataset with 500 rows for predicting car prices.
Features include mileage, year, fuel type, and engine size.
Target is price, which increases with year and decreases with mileage."

GOOD tool input (classification):
"Create a classification dataset with 1000 rows for predicting customer churn.
Features include age, monthly usage, contract type, and number of support calls.
Target is churn (yes/no) with roughly balanced classes."

GOOD tool input (clustering):
"Create a clustering dataset with 800 rows for customer segmentation.
Features include income, spending score, and age.
Generate 4 distinct clusters."

-----------------------
BEHAVIOR RULES
-----------------------

- Prefer making reasonable assumptions instead of asking too many questions
- Keep tool input concise but complete
- Ensure clarity and structure in the instruction
- Only call the tool when confident the dataset can be generated

Be helpful, precise, and efficient.

Assume when the tool is called, the user is automatically displayed the dataset preview and an option to download the dataset.
"""


In [ ]:
# ========== chat()：多轮对话 + 循环处理 tool_calls ==========

from google.colab import userdata
# 再取一次密钥（本格可单独重跑）
key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=key)
# 聊天大脑用的模型 id（字符串勿改）
MODEL = "gpt-4.1-mini"

def chat(message, history):
    # Gradio messages 格式 → 只保留 role/content 喂给 API
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # system + 历史 + 本轮 user
    messages = [{"role": "system", "content": BRAIN_SYSTEM_PROMPT}] + history + [{"role": "user", "content": message}]

    # 第一次调用：带上 tools，模型可能直接答或发起 tool_calls
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    # 若工具被调用，从 tool 结果里抽出预览与文件路径
    preview = None
    file_path = None

    # 只要 finish_reason 仍是 tool_calls，就执行工具并把结果追加回 messages
    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        responses = handle_tool_calls(message_obj)

        # 从 tool 返回的 JSON 里取出 preview / file_path 给 UI
        for r in responses:
            content = json.loads(r["content"])
            preview = content.get("preview")
            file_path = content.get("file_path")

        # 协议：先 append 含 tool_calls 的 assistant message，再 append 各 tool 结果
        messages.append(message_obj)
        messages.extend(responses)

        # 带着工具结果再问模型，让它生成最终自然语言回复
        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    final_text = response.choices[0].message.content

    return final_text, preview, file_path


In [ ]:
# ========== 把 preview dict 转成 DataFrame，方便 Gradio 表格展示 ==========

import gradio as gr
import pandas as pd

def format_preview(preview):
    # 没有数据时返回 None，界面可隐藏
    if preview is None:
        return None
    return pd.DataFrame(preview)


In [ ]:
# ========== Gradio Blocks：聊天 + 条件显示预览/下载 ==========

with gr.Blocks() as demo:
    # type="messages"：使用 role/content 消息列表格式
    chatbot = gr.Chatbot(type="messages")
    msg = gr.Textbox(label="Enter your request")

    # 初始隐藏：只有真正生成了数据集才显示
    preview_df = gr.Dataframe(label="Dataset Preview", visible=False)
    file_output = gr.File(label="Download Dataset", visible=False)

    def respond(message, history):
      # 调 chat：拿回复文本、预览、文件路径
      text, preview, file_path = chat(message, history)

      # 把本轮 user/assistant 写回 history
      history.append({"role": "user", "content": message})
      history.append({"role": "assistant", "content": text})

      if file_path:  # ✅ tool was called
          # 有文件：显示预览表与下载
          return (
              history,
              gr.update(value=format_preview(preview), visible=True),
              gr.update(value=file_path, visible=True)
          )
      else:  # ❌ no dataset generated
          # 仅澄清问题、未调用工具：隐藏预览与下载
          return (
              history,
              gr.update(visible=False),
              gr.update(visible=False)
          )

    # 回车提交：输入框 + 聊天历史 → 更新三块输出
    msg.submit(respond, [msg, chatbot], [chatbot, preview_df, file_output])

demo.launch()


## 用开源模型做带工具的 LLM——JSON 输出

后半段：换 **Hugging Face** 本地/Colab 开源 Instruct 模型，直接用同一套 `system_message` 让模型吐出数据集 JSON spec，对比 Gemma / Phi / Qwen / DeepSeek / Llama 的服从度。需要 GPU 与 `HF_TOKEN`。


In [ ]:
# ========== 安装 HF 推理依赖（量化 + transformers 固定版本） ==========
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6


In [ ]:
# ========== 候选开源模型 id（字符串是 Hub 路径，勿改） ==========

LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"
PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"


In [ ]:
# ========== 导入：HF 登录、分词器/模型、量化配置、torch、垃圾回收 ==========

from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import gc


In [ ]:
# ========== 登录 Hugging Face：拉取 gated 模型需要 token ==========

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)


In [ ]:
# ========== 4-bit 量化配置：让 7B/8B 级模型更吃得进显存 ==========

quant_config = BitsAndBytesConfig(
    # 权重量化到 4 bit
    load_in_4bit=True,
    # 双重量化进一步省内存
    bnb_4bit_use_double_quant=True,
    # 计算 dtype：bfloat16
    bnb_4bit_compute_dtype=torch.bfloat16,
    # nf4 量化类型
    bnb_4bit_quant_type="nf4"
)


In [ ]:
# ========== 构造测试 messages：同一条聚类请求对比各开源模型 ==========

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": """Create a clustering dataset with 800 rows for customer segmentation.
Features include income, spending score, gender,City, age.
Generate 4 distinct clusters."""}
  ]


In [ ]:
# ========== 带缓存的模型加载：同 (name, quant) 只 from_pretrained 一次 ==========

MODEL_CACHE = {}

def load_model(model_name, quant=True):
    key = (model_name, quant)

    # 命中缓存直接返回
    if key in MODEL_CACHE:
        return MODEL_CACHE[key]

    print(f"🔄 Loading model: {model_name} | quant={quant}")

    # use_fast=False：部分模型快分词器不稳定时更稳妥
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    # 补 pad_token，避免 generate 警告/错误
    tokenizer.pad_token = tokenizer.eos_token

    if quant:
        # 走 bitsandbytes 4-bit
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config,
            device_map="auto"
        )
    else:
        # 不量化（小模型如 Gemma 270M 可直接加载）
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto"
        )

    MODEL_CACHE[key] = (tokenizer, model)
    return tokenizer, model


In [ ]:
# ========== 生成：chat template 编码 → generate → 只解码新 token ==========

def generate(model_name, messages, quant=True, max_new_tokens=1024):
    # 取分词器与模型（可能触发首次加载）
    tokenizer, model = load_model(model_name, quant)

    # 把 messages 编成该模型的对话模板张量，并搬到模型设备
    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to(model.device)

    # 全 1 attention mask
    attention_mask = torch.ones_like(input_ids)

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id
    )

    # 去掉 prompt 前缀，只保留新生成部分
    generated_tokens = outputs[0][input_ids.shape[-1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response


In [ ]:
# 对比 1：Gemma 小模型，不量化
print(generate(GEMMA, messages, quant=False))


In [ ]:
# 对比 2：Phi-4-mini，4-bit 量化
print(generate(PHI, messages, quant=True))


In [ ]:
# 对比 3：Qwen3-4B Instruct，4-bit 量化
print(generate(QWEN, messages, quant=True))


In [ ]:
# 对比 4：DeepSeek-R1 蒸馏小模型，4-bit 量化
print(generate(DEEPSEEK, messages, quant=True))


In [ ]:
# 对比 5：Llama-3.1-8B Instruct，4-bit 量化（通常最吃显存）
print(generate(LLAMA, messages, quant=True))
